# Classical Mechanics ODEs

This notebook explores eight fundamental ordinary differential equations from classical mechanics, each registered in the `diff_eq_solver` package. For every equation we:

1. Retrieve the equation from the registry and display its symbolic (closed-form) solution.
2. Solve it numerically with given initial conditions and plot the time-series solution.
3. Produce phase portraits or other specialised visualisations where they are physically insightful.

The eight equations covered are:

| # | Equation | Key physics |
|---|----------|-------------|
| 1 | Simple Harmonic Oscillator | Conservative oscillation |
| 2 | Damped Harmonic Oscillator | Underdamped / critically damped / overdamped regimes |
| 3 | Forced Harmonic Oscillator | Resonance and steady-state response |
| 4 | Simple Pendulum | Nonlinear restoring force |
| 5 | Duffing Oscillator | Nonlinear stiffness and chaos |
| 6 | Van der Pol Oscillator | Self-sustained oscillations with limit cycles |
| 7 | Kepler Problem | Orbital mechanics and closed orbits |
| 8 | Euler Rigid Body | Torque-free rotation of a rigid body |


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from diff_eq_solver import registry
from diff_eq_solver import plot_ode_solution, plot_phase_portrait, plot_comparison, plot_orbit

from IPython.display import Math, display

print('Available equations:')
for name in registry.list_equations():
    print(f'  - {name}')

---

## 1. Simple Harmonic Oscillator

The simple harmonic oscillator is the canonical second-order ODE of physics:

$$x'' + \omega^2 x = 0$$

Its general solution is $x(t) = A\cos(\omega t) + B\sin(\omega t)$, representing pure sinusoidal oscillation at angular frequency $\omega$. Energy is conserved and the phase portrait is an ellipse.

In [ ]:
eq = registry.get('simple_harmonic_oscillator')
sol_sym = eq.symbolic_solve(omega=1.0)

print('Symbolic solution:')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Numerical solution for three different angular frequencies
omegas = [0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, w in zip(axes, omegas):
    sol_num = eq.numerical_solve(initial_conditions=[1.0, 0.0], t_span=(0, 20), omega=w)
    if sol_num.numerical:
        t, y = sol_num.numerical
        ax.plot(t, y[0], label=f'x(t), $\\omega$={w}')
        ax.plot(t, y[1], label=f"x'(t), $\\omega$={w}", linestyle='--')
        ax.set_xlabel('t')
        ax.set_ylabel('x')
        ax.set_title(f'Simple Harmonic Oscillator ($\\omega={w}$)')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Phase portrait: x vs x' forms an ellipse for each omega
fig, ax = plt.subplots(figsize=(6, 6))
for w in omegas:
    sol_num = eq.numerical_solve(initial_conditions=[1.0, 0.0], t_span=(0, 20), omega=w)
    if sol_num.numerical:
        t, y = sol_num.numerical
        ax.plot(y[0], y[1], label=f'$\\omega={w}$')

ax.set_xlabel('x')
ax.set_ylabel("x'")
ax.set_title('Phase Portrait: Simple Harmonic Oscillator')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---

## 2. Damped Harmonic Oscillator

Adding a velocity-proportional damping term gives:

$$x'' + 2\gamma x' + \omega^2 x = 0$$

The dynamics depend on the ratio $\gamma/\omega$:
- **Underdamped** ($\gamma < \omega$): oscillations with exponential decay
- **Critically damped** ($\gamma = \omega$): fastest return to equilibrium without oscillation
- **Overdamped** ($\gamma > \omega$): slow exponential return, no oscillation

In [ ]:
eq = registry.get('damped_harmonic_oscillator')
sol_sym = eq.symbolic_solve(gamma=0.2, omega=1.0)

print('Symbolic solution (underdamped case):')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Compare the three damping regimes
omega_val = 1.0
cases = {
    'Underdamped ($\\gamma=0.2$)': {'gamma': 0.2, 'omega': omega_val},
    'Critically damped ($\\gamma=1.0$)': {'gamma': 1.0, 'omega': omega_val},
    'Overdamped ($\\gamma=3.0$)': {'gamma': 3.0, 'omega': omega_val},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, params in cases.items():
    sol_num = eq.numerical_solve(initial_conditions=[1.0, 0.0], t_span=(0, 20), **params)
    if sol_num.numerical:
        t, y = sol_num.numerical
        axes[0].plot(t, y[0], label=label)
        axes[1].plot(y[0], y[1], label=label)

axes[0].set_xlabel('t')
axes[0].set_ylabel('x(t)')
axes[0].set_title('Time Series: Damped Harmonic Oscillator')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('x')
axes[1].set_ylabel("x'")
axes[1].set_title('Phase Portrait: Damped Harmonic Oscillator')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Energy decay for the underdamped case
sol_num = eq.numerical_solve(initial_conditions=[1.0, 0.0], t_span=(0, 30), gamma=0.15, omega=1.0)
if sol_num.numerical:
    t, y = sol_num.numerical
    x, v = y[0], y[1]
    kinetic = 0.5 * v**2
    potential = 0.5 * 1.0**2 * x**2
    total = kinetic + potential

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(t, kinetic, label='Kinetic Energy', alpha=0.7)
    ax.plot(t, potential, label='Potential Energy', alpha=0.7)
    ax.plot(t, total, label='Total Energy', linewidth=2)
    ax.set_xlabel('t')
    ax.set_ylabel('Energy')
    ax.set_title('Energy Decay in Underdamped Oscillator ($\\gamma=0.15$, $\\omega=1.0$)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---

## 3. Forced Harmonic Oscillator

A damped oscillator driven by an external periodic force:

$$x'' + 2\gamma x' + \omega_0^2 x = F_0 \cos(\omega t)$$

When the driving frequency $\omega$ approaches the natural frequency $\omega_0$, the steady-state amplitude reaches a maximum -- this is **resonance**. The damping $\gamma$ controls the width and height of the resonance peak.

In [ ]:
eq = registry.get('forced_harmonic_oscillator')
sol_sym = eq.symbolic_solve(gamma=0.1, omega_0=1.0, F0=1.0, omega_drive=0.8)

print('Symbolic solution:')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Resonance: sweep the driving frequency around omega_0
gamma_val = 0.1
omega_0_val = 1.0
F0_val = 1.0

driving_freqs = [0.5, 0.8, 1.0, 1.2, 1.5]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for w_d in driving_freqs:
    sol_num = eq.numerical_solve(
        initial_conditions=[0.0, 0.0],
        t_span=(0, 80),
        gamma=gamma_val,
        omega_0=omega_0_val,
        F0=F0_val,
        omega_drive=w_d
    )
    if sol_num.numerical:
        t, y = sol_num.numerical
        label = f'$\\omega_d={w_d}$'
        axes[0].plot(t, y[0], label=label, alpha=0.8)
        # Phase portrait for last 30 time units (steady state)
        mask = t > 50
        axes[1].plot(y[0][mask], y[1][mask], label=label, alpha=0.8)

axes[0].set_xlabel('t')
axes[0].set_ylabel('x(t)')
axes[0].set_title('Forced Oscillator: Transient + Steady State')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('x')
axes[1].set_ylabel("x'")
axes[1].set_title('Steady-State Phase Portraits')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Resonance curve: steady-state amplitude vs driving frequency
freq_range = np.linspace(0.1, 3.0, 100)
amplitudes = []

for w_d in freq_range:
    sol_num = eq.numerical_solve(
        initial_conditions=[0.0, 0.0],
        t_span=(0, 150),
        gamma=gamma_val,
        omega_0=omega_0_val,
        F0=F0_val,
        omega_drive=w_d
    )
    if sol_num.numerical:
        t, y = sol_num.numerical
        # Use the last quarter of the signal for steady state
        steady = y[0][len(y[0])//4:]
        amplitudes.append((np.max(steady) - np.min(steady)) / 2.0)
    else:
        amplitudes.append(0.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(freq_range, amplitudes, linewidth=2)
ax.axvline(omega_0_val, color='r', linestyle='--', label=f'$\\omega_0={omega_0_val}$')
ax.set_xlabel('Driving frequency $\\omega_d$')
ax.set_ylabel('Steady-state amplitude')
ax.set_title(f'Resonance Curve ($\\gamma={gamma_val}$, $\\omega_0={omega_0_val}$)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 4. Simple Pendulum

The exact equation of motion for a simple pendulum:

$$\theta'' + \frac{g}{L}\sin(\theta) = 0$$

For small angles $\sin(\theta) \approx \theta$ and the equation reduces to a simple harmonic oscillator. At larger amplitudes the nonlinearity becomes important: the period increases with amplitude and the motion deviates from sinusoidal.

In [ ]:
eq = registry.get('simple_pendulum')
sol_sym = eq.symbolic_solve(g=9.81, L=1.0)

print('Symbolic solution:')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Compare nonlinear pendulum at different initial angles
g_val, L_val = 9.81, 1.0
angles_deg = [10, 45, 90, 150, 170]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for angle in angles_deg:
    theta0 = np.radians(angle)
    sol_num = eq.numerical_solve(
        initial_conditions=[theta0, 0.0],
        t_span=(0, 10),
        g=g_val,
        L=L_val
    )
    if sol_num.numerical:
        t, y = sol_num.numerical
        label = f'$\\theta_0={angle}^\\circ$'
        axes[0].plot(t, y[0], label=label)
        axes[1].plot(y[0], y[1], label=label)

axes[0].set_xlabel('t')
axes[0].set_ylabel('$\\theta(t)$')
axes[0].set_title('Pendulum: Effect of Initial Amplitude')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('$\\theta$')
axes[1].set_ylabel("$\\theta'$")
axes[1].set_title('Phase Portrait: Nonlinear Pendulum')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Period vs amplitude: compare exact numerical period with the small-angle approximation
amplitudes_deg = np.arange(5, 176, 5)
periods = []

for angle in amplitudes_deg:
    theta0 = np.radians(angle)
    sol_num = eq.numerical_solve(
        initial_conditions=[theta0, 0.0],
        t_span=(0, 20),
        g=g_val,
        L=L_val
    )
    if sol_num.numerical:
        t, y = sol_num.numerical
        # Find period by detecting zero crossings (positive-going)
        theta = y[0]
        crossings = []
        for i in range(1, len(theta) - 1):
            if theta[i - 1] < 0 and theta[i] >= 0:
                # Linear interpolation
                frac = -theta[i - 1] / (theta[i] - theta[i - 1])
                crossings.append(t[i - 1] + frac * (t[i] - t[i - 1]))
        if len(crossings) >= 2:
            periods.append(crossings[1] - crossings[0])
        else:
            periods.append(np.nan)
    else:
        periods.append(np.nan)

T_small = 2 * np.pi * np.sqrt(L_val / g_val)  # small-angle period

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(amplitudes_deg, periods, 'o-', markersize=3, label='Numerical period')
ax.axhline(T_small, color='r', linestyle='--', label=f'Small-angle: $2\\pi\\sqrt{{L/g}}$ = {T_small:.3f} s')
ax.set_xlabel('Initial amplitude (degrees)')
ax.set_ylabel('Period (s)')
ax.set_title('Pendulum Period vs Amplitude')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 5. Duffing Oscillator

The Duffing oscillator introduces a cubic nonlinearity and external forcing:

$$x'' + \delta x' + \alpha x + \beta x^3 = \gamma \cos(\omega t)$$

With appropriate parameter choices the Duffing oscillator can exhibit:
- **Regular periodic motion** (small forcing)
- **Period-doubling bifurcations**
- **Chaotic dynamics** (larger forcing with damping)

In [ ]:
eq = registry.get('duffing_oscillator')
sol_sym = eq.symbolic_solve(delta=0.1, alpha=-1.0, beta=1.0, gamma_force=0.3, omega=1.2)

print('Symbolic solution:')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Periodic regime: small forcing on a double-well potential
sol_num = eq.numerical_solve(
    initial_conditions=[1.0, 0.0],
    t_span=(0, 60),
    delta=0.1,
    alpha=-1.0,
    beta=1.0,
    gamma_force=0.3,
    omega=1.2
)

if sol_num.numerical:
    t, y = sol_num.numerical
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(t, y[0])
    axes[0].set_xlabel('t')
    axes[0].set_ylabel('x(t)')
    axes[0].set_title('Duffing Oscillator: Periodic Regime')
    axes[0].grid(True, alpha=0.3)

    # Phase portrait (steady state only)
    mask = t > 30
    axes[1].plot(y[0][mask], y[1][mask], linewidth=0.5)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel("x'")
    axes[1].set_title('Duffing: Steady-State Phase Portrait')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Chaotic regime: stronger forcing
sol_num = eq.numerical_solve(
    initial_conditions=[1.0, 0.0],
    t_span=(0, 200),
    delta=0.25,
    alpha=-1.0,
    beta=1.0,
    gamma_force=0.4,
    omega=1.0
)

if sol_num.numerical:
    t, y = sol_num.numerical
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(t, y[0], linewidth=0.4)
    axes[0].set_xlabel('t')
    axes[0].set_ylabel('x(t)')
    axes[0].set_title('Duffing Oscillator: Chaotic Regime')
    axes[0].grid(True, alpha=0.3)

    # Poincare section: sample at the driving period
    T_drive = 2 * np.pi / 1.0
    sample_times = np.arange(0, t[-1], T_drive)
    x_samples = np.interp(sample_times, t, y[0])
    v_samples = np.interp(sample_times, t, y[1])
    # Skip transient
    skip = len(sample_times) // 3
    axes[1].scatter(x_samples[skip:], v_samples[skip:], s=2, alpha=0.6, color='darkblue')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel("x'")
    axes[1].set_title('Poincare Section (stroboscopic at $\\omega$)')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---

## 6. Van der Pol Oscillator

The Van der Pol oscillator models self-sustained oscillations with nonlinear damping:

$$x'' - \mu(1 - x^2)x' + x = 0$$

For $\mu > 0$ the system has a stable **limit cycle**: trajectories from any initial condition (except the origin) spiral outward (or inward) until they settle onto a closed orbit. The parameter $\mu$ controls the strength of the nonlinear damping and the shape of the limit cycle.

In [ ]:
eq = registry.get('van_der_pol_oscillator')
sol_sym = eq.symbolic_solve(mu=1.0)

print('Symbolic solution:')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Van der Pol for three values of mu
mu_values = [0.5, 2.0, 5.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for col, mu_val in enumerate(mu_values):
    # From a small initial condition (spiral out to limit cycle)
    sol_num = eq.numerical_solve(
        initial_conditions=[0.5, 0.0],
        t_span=(0, 50),
        mu=mu_val
    )
    if sol_num.numerical:
        t, y = sol_num.numerical
        axes[0, col].plot(t, y[0])
        axes[0, col].set_xlabel('t')
        axes[0, col].set_ylabel('x(t)')
        axes[0, col].set_title(f'Van der Pol: $\\mu={mu_val}$')
        axes[0, col].grid(True, alpha=0.3)

        axes[1, col].plot(y[0], y[1], linewidth=0.5)
        axes[1, col].set_xlabel('x')
        axes[1, col].set_ylabel("x'")
        axes[1, col].set_title(f'Phase Portrait ($\\mu={mu_val}$)')
        axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Limit cycle convergence: overlay trajectories from different initial conditions
mu_val = 2.0
ics = [
    (0.1, 0.0),
    (3.0, 0.0),
    (0.0, 4.0),
    (-2.0, 2.0),
]

fig, ax = plt.subplots(figsize=(7, 7))
for x0, v0 in ics:
    sol_num = eq.numerical_solve(
        initial_conditions=[x0, v0],
        t_span=(0, 100),
        mu=mu_val
    )
    if sol_num.numerical:
        t, y = sol_num.numerical
        ax.plot(y[0], y[1], linewidth=0.5, label=f'IC=({x0}, {v0})')

ax.set_xlabel('x')
ax.set_ylabel("x'")
ax.set_title(f'Van der Pol Limit Cycle Attraction ($\\mu={mu_val}$)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---

## 7. Kepler Problem

The Kepler problem describes the motion of a body under an inverse-square gravitational force. In polar coordinates the radial equation reduces to:

$$r'' - \frac{L^2}{r^3} = -\frac{\mu_{\\text{grav}}}{r^2}$$

where $L$ is the angular momentum per unit mass and $\mu_{\\text{grav}}$ is the gravitational parameter. The solutions are conic sections: ellipses for bound orbits ($E < 0$), parabolas ($E = 0$), and hyperbolae ($E > 0$).

In [ ]:
eq = registry.get('kepler_problem')
sol_sym = eq.symbolic_solve(mu_grav=1.0, L=1.0)

print('Symbolic solution:')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Elliptical orbit: bound orbit with different eccentricities
mu_grav_val = 1.0
L_val = 1.0

# Initial conditions chosen to give different eccentricities
# For a Kepler orbit, r(0) = r_periapsis, r'(0) = 0, with L = r^2 * theta'
orbit_configs = [
    {'label': 'Nearly circular (e~0)', 'r0': 0.5, 'rdot0': 0.0, 'theta_dot0': 2.0},
    {'label': 'Moderate eccentricity (e~0.5)', 'r0': 0.3, 'rdot0': 0.0, 'theta_dot0': 2.0},
    {'label': 'High eccentricity (e~0.9)', 'r0': 0.15, 'rdot0': 0.0, 'theta_dot0': 2.0},
]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

for cfg in orbit_configs:
    r0 = cfg['r0']
    theta_dot0 = cfg['theta_dot0']
    L_eff = r0**2 * theta_dot0

    sol_num = eq.numerical_solve(
        initial_conditions=[r0, 0.0, 0.0, theta_dot0],
        t_span=(0, 30),
        mu_grav=mu_grav_val,
        L=L_eff
    )
    if sol_num.numerical:
        t, y = sol_num.numerical
        r = y[0]
        theta = y[2] if len(y) > 2 else np.linspace(0, 2 * np.pi, len(r))
        ax.plot(theta, r, label=cfg['label'], linewidth=1.5)

ax.set_title('Kepler Orbits: Effect of Eccentricity')
ax.legend(loc='lower right')
ax.set_rmax(2.5)
plt.tight_layout()
plt.show()

In [ ]:
# Verify orbit closure: conservation of energy and angular momentum
sol_num = eq.numerical_solve(
    initial_conditions=[0.5, 0.0, 0.0, 2.0],
    t_span=(0, 50),
    mu_grav=mu_grav_val,
    L=0.5
)

if sol_num.numerical:
    t, y = sol_num.numerical
    r = y[0]
    rdot = y[1]
    theta_dot = y[3] if len(y) > 3 else np.ones_like(r)

    # Effective angular momentum
    L_actual = r**2 * theta_dot
    # Energy
    E = 0.5 * rdot**2 + 0.5 * L_actual**2 / r**2 - mu_grav_val / r

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(t, L_actual, label='$L = r^2 \\dot{\\theta}$')
    axes[0].set_xlabel('t')
    axes[0].set_ylabel('Angular momentum')
    axes[0].set_title('Angular Momentum Conservation')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t, E, label='E', color='orange')
    axes[1].set_xlabel('t')
    axes[1].set_ylabel('Total energy')
    axes[1].set_title('Energy Conservation')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---

## 8. Euler Rigid Body Equations

The torque-free rotation of a rigid body is governed by Euler's equations:

$$I_1 \omega_1' = (I_2 - I_3)\omega_2 \omega_3$$
$$I_2 \omega_2' = (I_3 - I_1)\omega_3 \omega_1$$
$$I_3 \omega_3' = (I_1 - I_2)\omega_1 \omega_2$$

The stability of rotation about each principal axis depends on the ordering of the moments of inertia. Rotation about the axis with the largest or smallest moment of inertia is stable, while rotation about the intermediate axis is unstable (the **tennis racket theorem** or **Dzhanibekov effect**).

In [ ]:
eq = registry.get('euler_rigid_body')
sol_sym = eq.symbolic_solve(I1=1.0, I2=2.0, I3=3.0)

print('Symbolic solution:')
if sol_sym.latex:
    display(Math(sol_sym.latex))
else:
    print(sol_sym.expression)

In [ ]:
# Stable rotation about axis with smallest moment of inertia (I1)
I1_val, I2_val, I3_val = 1.0, 2.0, 3.0

sol_num = eq.numerical_solve(
    initial_conditions=[5.0, 0.1, 0.1],  # mostly omega_1
    t_span=(0, 30),
    I1=I1_val,
    I2=I2_val,
    I3=I3_val
)

if sol_num.numerical:
    t, y = sol_num.numerical

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(t, y[0], label='$\\omega_1$')
    axes[0].plot(t, y[1], label='$\\omega_2$')
    axes[0].plot(t, y[2], label='$\\omega_3$')
    axes[0].set_xlabel('t')
    axes[0].set_ylabel('Angular velocity')
    axes[0].set_title('Stable Rotation about $I_1$ (smallest)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot on the angular momentum sphere (|L| = const)
    L1 = I1_val * y[0]
    L2 = I2_val * y[1]
    L3 = I3_val * y[2]
    axes[1].plot(L1, L2, linewidth=0.5)
    axes[1].set_xlabel('$L_1 = I_1 \\omega_1$')
    axes[1].set_ylabel('$L_2 = I_2 \\omega_2$')
    axes[1].set_title('Polhode: $L_1$ vs $L_2$')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Unstable rotation about the intermediate axis (I2): tennis racket theorem
sol_num = eq.numerical_solve(
    initial_conditions=[0.1, 5.0, 0.1],  # mostly omega_2
    t_span=(0, 30),
    I1=I1_val,
    I2=I2_val,
    I3=I3_val
)

if sol_num.numerical:
    t, y = sol_num.numerical

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(t, y[0], label='$\\omega_1$')
    axes[0].plot(t, y[1], label='$\\omega_2$')
    axes[0].plot(t, y[2], label='$\\omega_3$')
    axes[0].set_xlabel('t')
    axes[0].set_ylabel('Angular velocity')
    axes[0].set_title('Unstable Rotation about $I_2$ (intermediate)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Polhode on the energy ellipsoid
    T_rot = 0.5 * (I1_val * y[0]**2 + I2_val * y[1]**2 + I3_val * y[2]**2)
    axes[1].plot(t, T_rot, label='Kinetic energy $T$')
    L_sq = (I1_val * y[0])**2 + (I2_val * y[1])**2 + (I3_val * y[2])**2
    axes[1].plot(t, L_sq, label='$|L|^2$')
    axes[1].set_xlabel('t')
    axes[1].set_title('Conservation Laws')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Stable rotation about axis with largest moment of inertia (I3)
sol_num = eq.numerical_solve(
    initial_conditions=[0.1, 0.1, 5.0],  # mostly omega_3
    t_span=(0, 30),
    I1=I1_val,
    I2=I2_val,
    I3=I3_val
)

if sol_num.numerical:
    t, y = sol_num.numerical

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(t, y[0], label='$\\omega_1$')
    ax.plot(t, y[1], label='$\\omega_2$')
    ax.plot(t, y[2], label='$\\omega_3$')
    ax.set_xlabel('t')
    ax.set_ylabel('Angular velocity')
    ax.set_title('Stable Rotation about $I_3$ (largest)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---

## Summary

In this notebook we explored eight classical mechanics ODEs using the `diff_eq_solver` package:

1. **Simple Harmonic Oscillator** -- pure sinusoidal oscillation, elliptical phase portraits
2. **Damped Harmonic Oscillator** -- three damping regimes, energy dissipation
3. **Forced Harmonic Oscillator** -- resonance curves, transient vs steady state
4. **Simple Pendulum** -- nonlinearity and amplitude-dependent period
5. **Duffing Oscillator** -- periodic and chaotic regimes, Poincare sections
6. **Van der Pol Oscillator** -- limit cycles and nonlinear self-excitation
7. **Kepler Problem** -- elliptical orbits and conservation laws
8. **Euler Rigid Body** -- stable/unstable rotation axes, the tennis racket theorem

Each equation was solved both symbolically (where a closed-form solution exists) and numerically, with phase portraits and specialised visualisations revealing the qualitative dynamics.